In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc samplomatic
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Démarrage rapide avec Executor

<Accordion>
<AccordionItem title="Versions des packages">

Le code sur cette page a été développé en utilisant les exigences suivantes.
Nous recommandons d'utiliser ces versions ou des versions plus récentes.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
samplomatic~=0.18.0
```
</AccordionItem>
</Accordion>

Similaire à la primitive [Sampler](/guides/get-started-with-sampler), Executor échantillonne les registres de sortie des exécutions de circuits quantiques, mais il n'a pas de suppression ou d'atténuation d'erreurs intégrée. Au lieu de cela, il fait partie du [modèle d'exécution dirigée](/guides/directed-execution-model) qui fournit les ingrédients pour capturer les intentions de conception côté client, et déplace la génération coûteuse de variantes de circuits côté serveur. Executor suit les directives fournies dans les annotations et les options de circuit, génère et lie les valeurs de paramètres, exécute les circuits liés sur le matériel, et renvoie les résultats d'exécution et les métadonnées. Il ne prend aucune décision implicite pour toi et te donne un contrôle et une transparence totale.

> **Note:** Le package Qiskit n'a pas encore de classe de base pour la primitive Executor.

## Avant de commencer
Certains des exemples de code sur cette page utilisent `samplex`, qui fait partie du package Samplomatic. Par conséquent, avant d'exécuter ces blocs de code, tu dois installer Samplomatic, comme indiqué dans le bloc de code suivant. Pour plus d'informations, consulte la [documentation Samplomatic](https://qiskit.github.io/samplomatic).

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, Executor
from qiskit_ibm_runtime.quantum_program import QuantumProgram
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from samplomatic.transpiler import generate_boxing_pass_manager
from samplomatic import build

# Initialize the service and choose a backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

In [2]:
print(backend)

<IBMBackend('ibm_fez')>


### 2. Create and transpile a circuit

You need at least one circuit to use the Executor primitive.  It can optionally have parameters.

In [3]:
# Generate the circuit
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.h(1)
circuit.cz(0, 1)
circuit.h(1)

# Using `measure_all` automatically creates the necessary
# classical registers.
circuit.measure_all()

The circuit needs to be transformed to only use instructions supported by the QPU (referred to as *instruction set architecture (ISA)* circuits). Use the transpiler to do this.

In [4]:
# Transpile the circuit
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=0
)
isa_circuit = preset_pass_manager.run(circuit)

### 2. Créer et transpiler un circuit
Tu as besoin d'au moins un circuit pour utiliser la primitive Executor. Il peut optionnellement avoir des paramètres.

In [5]:
# Initialize an empty program
program = QuantumProgram(shots=25)

# Append the circuit to the program
program.append_circuit_item(isa_circuit)

Le circuit doit être transformé pour n'utiliser que des instructions prises en charge par la QPU (appelées circuits *d'architecture d'ensemble d'instructions (ISA)*). Utilise le Transpiler pour ce faire.

In [6]:
# Generate a boxing pass manager to group gates
# and measurements into boxes and add
# a`Twirl` annotation.
boxes_pm = generate_boxing_pass_manager(
    # Add gate twirling
    enable_gates=True,
    # Add measurement twirling
    enable_measures=True,
)

boxed_circuit = boxes_pm.run(isa_circuit)
boxed_circuit.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/get-started-with-executor/extracted-outputs/245a4574-3ce9-4f77-98c8-af32cde8ac01-0.svg" alt="Output of the previous code cell" />

### 3. Initialiser un `QuantumProgram`
Initialise un `QuantumProgram` avec ta charge de travail. Un `QuantumProgram` est composé d'éléments `QuantumProgramItems`. En général, chaque élément consiste en un circuit, un ensemble de valeurs de paramètres et éventuellement un `samplex` pour randomiser le contenu du circuit. Pour tous les détails, voir [Entrées et sorties Executor](/guides/executor-input-output).

La cellule suivante initialise un `QuantumProgram` et spécifie d'effectuer 25 shots. Ensuite, elle ajoute le circuit cible transpilé.

In [7]:
# Build the template circuit and the samplex
template_circuit, samplex = build(boxed_circuit)

# Append the template circuit and samplex as a `samplex_item`
program.append_samplex_item(
    template_circuit,
    samplex=samplex,
    shape=(num_randomizations := 20,),
)

### 4. Facultatif : Regrouper les gates et les mesures dans des boîtes annotées
Regrouper les instructions dans des boîtes et les annoter est le principal moyen de spécifier ton intention. Dans l'exemple suivant, nous utilisons `generate_boxing_pass_manager` et ses paramètres de torsion pour regrouper les gates à deux qubits et les mesures dans des boîtes et appliquer l'annotation de torsion.

In [8]:
# Initialize an Executor with the default options
executor = Executor(mode=backend)

# Submit the job
job = executor.run(program)
job

<RuntimeJobV2('d8286580bvlc73d1vmsg', 'executor')>

In [9]:
# Retrieve the result
result = job.result()

### 6. Invoquer Executor et obtenir les résultats
Exécute le `QuantumProgram` sur un Backend IBM&reg; en utilisant la primitive `Executor` avec les options par défaut. Consulte [Options Executor](/guides/executor-options) pour en savoir plus sur les options disponibles.